# Computational Exercises 4

## Exercise 1

The code simply generates the mesh and solves the VF.

### Mesh

The mesh is created separating each boundary and holes in different physical groups. Then, based on a parameter $r$, we set the size of the mesh and generate it.

### Solving VF

We proceed as the previous codes. Initialize the needed variables, create our function space, the BCs and the bilinear and linear forms.

The next code has the functions we'll use to the end of this assignment. They're basically an encapsulation of the original code

In [1]:
import firedrake as fd
from firedrake.output import VTKFile 
from firedrake.__future__ import interpolate 
from pathlib import Path

import gmsh
import numpy as np

# User defined data
bottom_wall = 0
right_wall  = 1
top_wall    = 2
left_wall   = 3
hole1_wall  = 4
hole2_wall  = 5
hole3_wall  = 6

inclusion_marker = 3
background_marker = 2

ninclusions = 3
Lx = 2.0
Ly = 2.0
R1 = 0.25
R2 = 0.15
R3 = 0.25

gdim = 2

#--------------------------------------------------------------------
#--- Preprocess: Mesh generation, boundary and region identification

def GenerateMesh():

    gmsh.initialize()

    # We create one rectangle and the circular inclusion
    rectangle = gmsh.model.occ.addRectangle(0, 0, 0, Lx, Ly)
    hole1 = gmsh.model.occ.addDisk(0.5, 1.0, 0, R1, R1)
    hole2 = gmsh.model.occ.addDisk(1.0, 1.5, 0, R2, R2)
    hole3 = gmsh.model.occ.addDisk(1.5, 1.0, 0, R3, R3)
    gmsh.model.occ.synchronize()
    all_holes = [(2, hole1)]
    all_holes.extend([(2, hole2)])
    all_holes.extend([(2, hole3)])
    whole_domain = gmsh.model.occ.cut([(gdim, rectangle)], all_holes)
    gmsh.model.occ.synchronize()
    background_surfaces = []
    for domain in whole_domain[0]:
        gmsh.model.addPhysicalGroup(domain[0], [domain[1]], tag=background_marker)
        background_surfaces.append(domain)
            
    # Tag the the different boundaries
    left = []
    right = []
    top = []
    bottom = []
    hole1,hole2,hole3 = [], [], []
    for line in gmsh.model.getEntities(dim=1):
        com = gmsh.model.occ.getCenterOfMass(line[0], line[1])
        if np.isclose(com[0], 0.0):
            #print('L', line, com)
            left.append(line[1])
        if np.isclose(com[0], Lx):
            #print('R', line, com)
            right.append(line[1])
        if np.isclose(com[1], 0.0):
            #print('B', line, com)
            bottom.append(line[1])
        if np.isclose(com[1], Ly):
            #print('T', line, com)
            top.append(line[1])
        if np.isclose(np.linalg.norm(com), np.sqrt((0.5)**2 + (1.0)**2)):
            #print('H1', line, com)
            hole1.append(line[1])
        elif np.isclose(np.linalg.norm(com), np.sqrt((1.0)**2 + (1.5)**2)) and np.isclose(com[1], 1.5):
            #print('H2', line, com)
            hole2.append(line[1])
        elif np.isclose(np.linalg.norm(com), np.sqrt((1.5)**2 + (1.0)**2)):
            #print('H3', line, com)
            hole3.append(line[1])
                
    gmsh.model.addPhysicalGroup(1, left, left_wall)
    gmsh.model.addPhysicalGroup(1, right, right_wall)
    gmsh.model.addPhysicalGroup(1, top, top_wall)
    gmsh.model.addPhysicalGroup(1, bottom, bottom_wall)
    gmsh.model.addPhysicalGroup(1, hole1, hole1_wall)
    gmsh.model.addPhysicalGroup(1, hole2, hole2_wall)
    gmsh.model.addPhysicalGroup(1, hole3, hole3_wall)
    gmsh.model.occ.synchronize()

    if(True):
        r = 0.01
        res_min = r
        res_max = 4 * r
        gmsh.model.mesh.field.add("Distance", 1)
        gmsh.model.mesh.field.setNumbers(1, "EdgesList", hole1+hole2+hole3)
        gmsh.model.mesh.field.add("Threshold", 2)
        gmsh.model.mesh.field.setNumber(2, "IField", 1)
        gmsh.model.mesh.field.setNumber(2, "LcMin", res_min)
        gmsh.model.mesh.field.setNumber(2, "LcMax", res_max)
        gmsh.model.mesh.field.setNumber(2, "DistMin", 2*r)
        gmsh.model.mesh.field.setNumber(2, "DistMax", 4*r)
        # We take the minimum of the two fields as the mesh size
        gmsh.model.mesh.field.add("Min", 5)
        gmsh.model.mesh.field.setNumbers(5, "FieldsList", [2])
        gmsh.model.mesh.field.setAsBackgroundMesh(2)
        # Generate mesh
        gmsh.option.setNumber("Mesh.Algorithm", 6)
        gmsh.model.mesh.generate(2)
    else:
        gmsh.model.mesh.setSize(gmsh.model.getEntities(0), 0.1)
        gmsh.model.mesh.generate(2)
        
    gmsh.write("mesh.msh")
    gmsh.finalize()
    mesh = fd.Mesh("mesh.msh")

    return mesh
#--------------------------------------------------------------------


def GenMesh():
    file_path = Path("mesh.msh")
    if file_path.exists():
        print("Mesh already exists")
        return fd.Mesh("mesh.msh")
        
    else:
        return GenerateMesh()

def solveVF(mesh, mu, gamma, source, uleft, uright, T1, T2, T3, ht, hb, VTK:str | None = None):


    Vh = fd.FunctionSpace(mesh, "CG", degree=1)

    # Set Dirchlet values
    bcs = []
    bcs.append(fd.DirichletBC(Vh, fd.Constant(uleft), left_wall))
    bcs.append(fd.DirichletBC(Vh, fd.Constant(uright), right_wall))
    bcs.append(fd.DirichletBC(Vh, T1, hole1_wall))
    bcs.append(fd.DirichletBC(Vh, T2, hole2_wall))
    bcs.append(fd.DirichletBC(Vh, T3, hole3_wall))


    # Variational formulation
    x = fd.SpatialCoordinate(mesh) 
    u, v = fd.TrialFunction(Vh), fd.TestFunction(Vh)

    a = fd.inner(mu*fd.grad(u), fd.grad(v))*fd.dx + fd.inner(gamma*u,v)*(fd.ds(bottom_wall) + fd.ds(top_wall))
    L = source * v * fd.dx + ht * v * fd.ds(top_wall) + hb * v * fd.ds(bottom_wall)

    # Solve the problem
    uh = fd.Function(Vh, name='Temperature')
    opts={"ksp_type": "preonly", "pc_type": "lu"}
    fd.solve(a==L, uh, bcs=bcs, solver_parameters=opts)
    if not(VTK is None):
        VTKFile(VTK).write(uh)

    return uh

firedrake:WARNING OMP_NUM_THREADS is not set or is set to a value greater than 1, we suggest setting OMP_NUM_THREADS=1 to improve performance


## Exercise 2

We must use the variables with values:
- $\mu = 1$
- $f = 0$
- $u_l = 100$
- $u_r = 1$
- $h_t = h_b = 1$

Using the results from previous assignment ($\mu_B = 10000$)
- $u_1 = 74.4103$
- $u_2 = 50.4995$
- $u_3 = 26.5897$ 

In [17]:

mesh = GenMesh()

# Material parameters and boundary values
mu = fd.Constant(1.0)
gamma = fd.Constant(0.0)
source = fd.Constant(0.0)
uleft = 100.0
uright = 1.0
T1 = fd.Constant(74.4103)
T2 = fd.Constant(50.4995)
T3 = fd.Constant(26.5897)
ht = fd.Constant(1.0)
hb = fd.Constant(1.0)

solveVF(mesh, mu, gamma, source, uleft, uright, T1, T2, T3, ht, hb, "Solutions/ex2.pvd")



Mesh already exists


Coefficient(WithGeometry(FunctionSpace(<firedrake.mesh.MeshTopology object at 0x7ff135f8fb10>, FiniteElement('Lagrange', triangle, 1), name=None), Mesh(VectorElement(FiniteElement('Lagrange', triangle, 1), dim=2), 573)), 1333)

The plots:

- Previous assignment

![Image from previous assignment](imgs/ex2_last.png)

- Plotted now

![Plot generated from code above](imgs/ex2.png)

They are qualitatively very close from one another. This is the expected result. With a large $\mu_B$, the temperature inside the inclusions are almost constant because the conductivity is high. So, it's intuitive that only the border of the inclusions would affect the PDE. In fact this occurs.

## Exercise 3

Now with $\gamma = 1$:

In [16]:
mesh = GenMesh()

# Material parameters and boundary values
mu = fd.Constant(1.0)
gamma = fd.Constant(1.0)
source = fd.Constant(0.0)
uleft = 100.0
uright = 1.0
T1 = fd.Constant(74.4103)
T2 = fd.Constant(50.4995)
T3 = fd.Constant(26.5897)
ht = fd.Constant(1.0)
hb = fd.Constant(1.0)

solveVF(mesh, mu, gamma, source, uleft, uright, T1, T2, T3, ht, hb, "Solutions/ex3.pvd")

Mesh already exists


Coefficient(WithGeometry(FunctionSpace(<firedrake.mesh.MeshTopology object at 0x7ff14ecaefd0>, FiniteElement('Lagrange', triangle, 1), name=None), Mesh(VectorElement(FiniteElement('Lagrange', triangle, 1), dim=2), 549)), 1280)

We obtain the following plot:

![Plot with Robin conditions](imgs/ex_3.png)

Notice that $-\mu \nabla u \cdot \check{n}$ is the outwards heat flux. If it's positive then heat leaves the domain and, the amount that leaves, is somehow proportional to it's magnitude. In the Robin condition:
$$
    -\mu \nabla u \cdot \check{n} = \gamma u - h_i , i = t, b
$$

So the flux is proportional to the value of $u$ at the top and bottom of the domain. In the left, we have a Dirichlet BC with a high value, so the more to the left, the higher the flux outwards through the bottom and top boundaries. This explains the plot having more areas with lower temperature compared to Exercise 2.

## Exercise 4

We'll run with the code with the following values:

- $\gamma \in \{10^{-5}, 0.1, 1.0, 10.0, 100.0, 1000.0\}$
- $f = 0$
- $\mu = 1$
- $u_l = u_r = 10$
- $u_i$ as before
- $h_t = h_b = \gamma u_l$

In [18]:
mesh = GenMesh()

# Material parameters and boundary values
mu = fd.Constant(1.0)
gamma_list = [1.0e-5, 0.1, 1.0, 10.0, 100.0, 1000.0]
source = fd.Constant(0.0)
uleft = 10.0
uright = 10.0
T1 = fd.Constant(74.4103)
T2 = fd.Constant(50.4995)
T3 = fd.Constant(26.5897)

for gamm in gamma_list:
    gamma = fd.Constant(gamm)
    ht = fd.Constant(gamm * uleft)
    hb = fd.Constant(gamm * uleft)
    solveVF(mesh, mu, gamma, source, uleft, uright, T1, T2, T3, ht, hb, f"Solutions/ex4_g={gamm}.pvd")

Mesh already exists


We got the following plots:

- $\gamma = 10^{-5}$

![](imgs/ex4_g=1e-5.png)

- $\gamma = 0.1$

![](imgs/ex4_g=0.1.png)

- $\gamma = 1.0$

![](imgs/ex4_g=1.png)

- $\gamma = 10.0$

![](imgs/ex4_g=10.png)

- $\gamma = 100.0$

![](imgs/ex4_g=100.png)

- $\gamma = 1000.0$

![](imgs/ex4_g=1000.png)

With smaller $\gamma$, the heat can flow out the domain better. Using the same development as the previous exercise and using $h_i = \gamma u_l$, we obtain:
$$
    -\mu \nabla u \cdot \check{n} = \gamma u - h_i = \gamma(u - u_l) , i = t, b
$$

So the outwards flux at a point is proportional to the difference of $u$ at the point and $u_l$. In general, $u > u_l$, because the temperature at the inclusions boundaries are higher than $u_l$. As the temperature of the first inclusion is the highest, points closer to it at the bottom or top boundaries have a greater flux. Now, addressing the value of $\gamma$, when gamma is small, the outwards flux is low, so the heat gets trapped and we see higher temperatures. When $\gamma$ is great, the flux grows and the heat scapes, which results in smaller temperatures.

## Exercise 5

We just make $u_l = 100$

In [19]:
mesh = GenMesh()

# Material parameters and boundary values
mu = fd.Constant(1.0)
gamma_list = [1.0e-5, 0.1, 1.0, 10.0, 100.0, 1000.0]
source = fd.Constant(0.0)
uleft = 100.0
uright = 10.0
T1 = fd.Constant(74.4103)
T2 = fd.Constant(50.4995)
T3 = fd.Constant(26.5897)

for gamm in gamma_list:
    gamma = fd.Constant(gamm)
    ht = fd.Constant(gamm * uleft)
    hb = fd.Constant(gamm * uleft)
    solveVF(mesh, mu, gamma, source, uleft, uright, T1, T2, T3, ht, hb, f"Solutions/ex5_g={gamm}.pvd")

Mesh already exists


We got the following plots:

- $\gamma = 10^{-5}$

![](imgs/ex5_g=1e-5.png)

- $\gamma = 0.1$

![](imgs/ex5_g=0.1.png)

- $\gamma = 1.0$

![](imgs/ex5_g=1.png)

- $\gamma = 10.0$

![](imgs/ex5_g=10.png)

- $\gamma = 100.0$

![](imgs/ex5_g=100.png)

- $\gamma = 1000.0$

![](imgs/ex5_g=1000.png)


Now we have $u_l > u$ at the top and bottom boundaries. So $u- u_l < 0$. That means heat is entering the domain. As $\gamma$ grows, the amount of heat that enters increases. It also increases when u is smallest, that is, more to the right, closer to the Dirichlet BC with value $10.0$. At the right bottom and upper corners, when $\gamma$ is big, we can see a spot where the temperature is very high, favoring the stated point.

As $\gamma$ gets smaller, we approach a Neumann BC at the bottom and top boundaries. We basically recover the Exercise 2 solution 

## Exercise 6

Now the constants and BCs are:

- $f = 0$
- $\gamma = 0$
- $\mu = 1$
- $u_l = u_r = 0$
- $h_t = h_b =  0.2x_1(L_x - x_1)$
- $u_i = 0.1\sin^2[ (i+2) \theta_i]$ with $\theta_i = \arctan\left( \frac{x_2 - x^i_{c2}}{x_1 - x^i_{c1}} \right)$, $i = 1, 2, 3$

In [3]:
mesh = GenMesh()

x = fd.SpatialCoordinate(mesh)

def ui(y, i, c1, c2):
    theta = fd.atan((y[1]- c2)/(y[0] - c1))
    return fd.Constant(0.1) * fd.sin((i+2) * theta) * fd.sin((i+2) * theta)

# Material parameters and boundary values
mu = fd.Constant(1.0)
gamma = fd.Constant(0.0)
source = fd.Constant(0.0)
uleft = 0.0
uright = 0.0
ht = fd.Constant(0.2) * x[0] * (Lx - x[0])
hb = fd.Constant(0.2) * x[0] * (Lx - x[0])

def T1(y):
    return ui(y, 1, 0.5, 1.0)
def T2(y):
    return ui(y, 2, 1.0, 1.5)
def T3(y): 
    return ui(y, 3, 1.5, 1.0)

Vh = fd.FunctionSpace(mesh, "CG", 1)

solveVF(mesh, mu, gamma, source, uleft, uright, T1(x), T2(x), T3(x), ht, hb, f"Solutions/ex_6.pvd")


Mesh already exists


Coefficient(WithGeometry(FunctionSpace(<firedrake.mesh.MeshTopology object at 0x7fc77fb8c050>, FiniteElement('Lagrange', triangle, 1), name=None), Mesh(VectorElement(FiniteElement('Lagrange', triangle, 1), dim=2), 3)), 25)

We obtain a plot that resembles the one in the lecture notes:

![](imgs/ex6.png)